# Задания к занятию №4

## Задание 1.1: Таинственный DataFrame

Дан код:
```
import pandas as pd

df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
print("Original ID:", id(df))

def clean_data(data):
    data = data[data['A'] > 1]  # фильтруем строки
    print("Inside function ID:", id(data))
    data.loc[:, 'C'] = data['A'] * 2  # добавляем колонку
    return data

new_df = clean_data(df)
print("New df ID:", id(new_df))
print("Original df after function:")
print(df)
print("New df:")
print(new_df)

```

Вопросы:

1) Сколько различных объектов DataFrame существует в памяти после выполнения кода?
2) Почему df остался неизменным, если DataFrame $-$ mutable объект?
3) Что произойдёт, если внутри функции всместо
   ```
   data = data[data['A'] > 1]
   ```
   
   написать:
   
   ```
   data.drop(index=data[data['A'] <= 1].index, inplace=True)?
   ```

In [4]:
import pandas as pd

df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
print("Original ID:", id(df))

def clean_data(data):
    data = data[data['A'] > 1]  # фильтруем строки
    print("Inside function ID:", id(data))
    data.loc[:, 'C'] = data['A'] * 2  # добавляем колонку
    return data

new_df = clean_data(df)
print("New df ID:", id(new_df))
print("Original df after function:")
print(df)
print("New df:")
print(new_df)

Original ID: 132074707448512
Inside function ID: 132074347073024
New df ID: 132074347073024
Original df after function:
   A  B
0  1  4
1  2  5
2  3  6
New df:
   A  B  C
1  2  5  4
2  3  6  6


/tmp/ipykernel_503/2200550542.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:, 'C'] = data['A'] * 2  # добавляем колонку


---

### 1 вопрос - 2 объекта (df и new_df старый и новый объект)


### 2 вопрос - мы никак не редактировали df и не изменяли (создали копию?)


### 3 вопрос - мы применили к нашей таблице изменения ```inplace=True``` то есть радактируем нашу основу, а не создаем копию (будет 1 объект)

---



## Задание 1.2: Списковое недоразумение

Объясните, почему следующий код работае не так, как мог бы ожидать новичок?

```
def add_and_multiply(numbers, multiplier=[]):
    multiplier.append(1)
    return [n * len(multiplier) for n in numbers]

print(add_and_multiply([1, 2, 3]))  # Ожидание: [1, 2, 3]?
print(add_and_multiply([1, 2, 3]))  # Ожидание снова [1, 2, 3]? А получаем...
print(add_and_multiply([1, 2, 3], [5, 5]))  # А тут?
```

Исправьте функцию так, чтобы она работала предсказуемо, но сохранила возможность передавать свой список `multiplier`.

In [1]:
def add_and_multiply(numbers, multiplier=[]):

    multiplier = multiplier.copy() # можно добавить копирование чтобы избежать влияния на следующие вызовы

    multiplier.append(1)
    return [n * len(multiplier) for n in numbers]

print(add_and_multiply([1, 2, 3]))
print(add_and_multiply([1, 2, 3]))
print(add_and_multiply([1, 2, 3], [5, 5]))

[1, 2, 3]
[1, 2, 3]
[3, 6, 9]


---


## Задание 2.1: Собственный OneHotEncoder

Создайте класс `SimpleOneHotEncoder`, который будет обучаться на списке категорий и преобразовывать новые данные.

Требования:
- Метод `fit(categories)` принимает список строк или `pandas.Series`;
- Метод `transform(categories)` возвращает pandas.DataFrame с one-hot закодированными колонками (порядок колонок должен соответствовать порядку уникальных категорий из `fit`);
- Метод `fit_transform(categories)` $-$ выполняет обе операции и возвращает результат.

**Важно**: Если на этапе `transform` встречается категория, которой не было в `fit`, метод должен выбросить понятное исключение `ValueError` с указанием "неизвестной категории".

#### Подсказки:
- храните уникальные значения в атрибуте `self.categories_`;
- подумайте о том, что произойдёт с памятью, если передать огромный список в `fit`;
- что такое one-hot encoding и зачем он нужен: <a href=https://sky.pro/wiki/analytics/one-hot-encoding-chto-eto-takoe-i-kak-primenyat-v-mashinnom-obuchenii/>тык</a>;
- про модель исключений в python: <a href=https://education.yandex.ru/handbook/python/article/model-isklyuchenij-python-try-except-else-finally-moduli>тык</a>.

In [ ]:
import pandas as pd
import numpy as np

arr = pd.Series(['blue','green', 'red', 'red'])

class SimpleOneHotEncoder:

  def __init__(self):
    self.categories_ = None

  def fit(self, data, sorting = True):

    self.categories_ = np.unique(data)
    if sorting:
      self.categories_.sort()

    return self

  def transform(self, data):
    if self.categories_ is None:
      raise ValueError("неизвестной категории")
    
    

    df = pd.DataFrame(self.categories_)
    df = df.T
    return df

  def fit_transform(self):
    return self.fit(self).transform(self)


In [57]:
OHE = SimpleOneHotEncoder()

OHE.fit(arr)
data = OHE.transform()
data

,0,1,2
0,blue,green,red


In [4]:
OHE_W = SimpleOneHotEncoder()

data_W = OHE_W.fit_transform(arr)
data_W

TypeError: SimpleOneHotEncoder.fit_transform() takes 1 positional argument but 2 were given

## Задание 3.1: Кэширующий трансформер

Представьте, что у вас есть "тяжёлая" функция предобработки текста (очистка, стемминг/лемматизация). Создайте класс `CachedTransformer`, который запоминает результаты обработки для повторяющихся входов.

In [ ]:
class CachedTransformer:
    def __init__(self, transformer_func):
        self.transformer_func = transformer_func
        self.cache = {}  # хранилище: вход -> результат

    def transform(self, X):
        """
        X: список строк
        Возвращает список обработанных строк.
        Если строка уже обрабатывалась, вернуть результат из кэша.
        """
        # Ваша реализация
        pass

**Вопросы**:
- Какие проблемы могут возникнуть с памятью, если кэшировать абсолютно все уникальные строки?
- Как можно ограничить размер кэша (например, хранить только 1 000 последних уникальных значений)? (ответ на этот вопрос реализовать в коде).

## Финальный проект: мини-проект "свой sklearn"

Создайте класс `DataFeatureExtractor`, который из колонки с датами генерирует набор числовых признаков.

In [ ]:
class DateFeatureExtractor:
    def __init__(self, date_column, features=['year', 'month', 'day', 'weekday']):
        """
        date_column: название колонки с датами (str)
        features: список признаков для генерации
        """
        self.date_column = date_column
        self.features = features
        self.fitted_ = False

    def fit(self, X, y=None):
        """
        X: pandas DataFrame
        Проверяет, что колонка существует и может быть преобразована в datetime.
        Вычисляет минимальную и максимальную дату (могут пригодиться).
        """
        # Ваш код
        pass

    def transform(self, X):
        """
        Возвращает НОВЫЙ DataFrame, содержащий только сгенерированные признаки.
        Оригинальный DataFrame не должен изменяться!
        """
        # Ваш код
        pass

**Дополнительные требования**:

1. Добавьте возможность генерации "циклических" признаков (`sin(month)`, `cos(month)`) для моделей, чувствительных к периодичнсоти;
2. Реализуйте метод `get_feature_names_out()`, возвращающий список названий созданных колонок;
3. Наследуйтесь от `sklearn.base.BaseEstimator` и `sklearn.base.TransformerMixin`, чтобы получить методы `get_params` и `set_params`, а также `fit_transform`.

In [ ]:
# здесь необходимо протестировать разработанный класс
# для генерации тестовых данных разрешается обращаться к нейросетям